# Faz 4 — Referans Perceptron, Ağaç Tabanlı GBDT ve Optuna Optimizasyonu

**Sorumlu:** Ertuğrul (Klasik ve Ağaç Modelleri Uzmanı — Kod Savunması)

Bu notebook proje şartnamesindeki Faz 4 gereksinimlerini uygular:
1. **Metrik Kuralı (`Accuracy` Kesinlikle Yasak):** Yüksek sınıf dengesizliği (`1:7.96`) nedeniyle `Accuracy` kullanılmaz; tüm değerlendirmeler yalnızca **`Macro F1-Score`**, **`Weighted F1-Score`**, **`ROC-AUC`** ve **`PR-AUC (Average Precision)`** üzerinden raporlanır.
2. **Model 1 — Referans (`Baseline`) Perceptron:** Farklı öğrenme oranları (`eta0`) ve sınıf ağırlıkları (`class_weight='balanced'` vs. `None`) yalnızca eğitim seti (`X_train`) üzerinde **5-Katlı Stratified Cross-Validation** ile taranır; `decision_function` çıktıları lojistik sigmoid (`expit`) ile `(0, 1)` olasılık/skor aralığına dönüştürülür.
3. **Model 2 — Ağaç Tabanlı GBDT (`LightGBM` / `XGBoost`) ve `Optuna`:** Ağaç tabanlı aday modeller karşılaştırılır ve `LightGBM` hiperparametreleri `Optuna` (`TPESampler`) + `StratifiedKFold` ile **yalnızca `X_train` üzerinde** (sıfır veri sızıntısı) optimize edilir.
4. **Özellik Önemleri (`feature_importances_`):** Optimize edilen GBDT modelinin hem `154` kodlanmış özellik hem de `42` ana klinik özellik düzeyindeki önem skorları çıkarılır.
5. **Faz 5 Entegrasyonu (`faz4_outputs.pkl`):** Faz 5'te Yunus Emre'nin çizeceği 4., 5. ve 6. zorunlu grafikler için `y_pred`, `y_prob`, normalize karmaşıklık matrisleri ve GBDT özellik önemleri `data/processed/faz4_outputs.pkl` dosyasına kaydedilir.

In [1]:
import time

from IPython.display import display
import pandas as pd

from src import config
from src import models

## 1. Faz 2'den İşlenmiş Eğitim ve Test Verilerinin Yüklenmesi

Faz 2'de sızıntısız `IterativeImputer / MICE` boru hattı ile dönüştürülen `X_train`, `X_test`, `y_train` ve `y_test` veri setleri yüklenir.

In [2]:
X_train = pd.read_pickle(config.PROCESSED_DATA_DIR / 'X_train.pkl')
X_test = pd.read_pickle(config.PROCESSED_DATA_DIR / 'X_test.pkl')
y_train = pd.read_pickle(config.PROCESSED_DATA_DIR / 'y_train.pkl')
y_test = pd.read_pickle(config.PROCESSED_DATA_DIR / 'y_test.pkl')
df_missing = pd.read_pickle(
    config.PROCESSED_DATA_DIR / 'df_missing.pkl'
)

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print(f'X_train : {X_train.shape} | Sınıf 0: '
      f'{int((y_train == 0).sum()):,} | Sınıf 1: '
      f'{int((y_train == 1).sum()):,} '
      f'(Pozitif oran: {y_train.mean():.4f})')
print(f'X_test  : {X_test.shape}  | Sınıf 0: '
      f'{int((y_test == 0).sum()):,} | Sınıf 1: '
      f'{int((y_test == 1).sum()):,} '
      f'(Pozitif oran: {y_test.mean():.4f})')

X_train : (81412, 154) | Sınıf 0: 72,326 | Sınıf 1: 9,086 (Pozitif oran: 0.1116)
X_test  : (20354, 154)  | Sınıf 0: 18,083 | Sınıf 1: 2,271 (Pozitif oran: 0.1116)


## 2. Model 1 — Referans (`Baseline`) Perceptron ve Öğrenme Oranı (`eta0`) Taraması

Şartname rubriğinde belirtilen *"Perceptron Baseline: Proper setup, learning rate adjustment, and baseline performance tracking"* gereksinimi doğrultusunda `eta0` değerleri (`0.0001` – `1.0`) ve sınıf ağırlığı (`class_weight='balanced'` vs. `None`) seçenekleri yalnızca `X_train` üzerinde **5-Katlı Stratified Cross-Validation** ile taranır.

In [3]:
t0 = time.perf_counter()
best_perceptron, perceptron_cv_df, best_perc_params = (
    models.tune_perceptron(X_train, y_train, n_splits=5)
)
perc_sec = time.perf_counter() - t0

print(f'Perceptron CV tarama süresi : {perc_sec:.2f} sn')
print(f'Seçilen en iyi parametreler : {best_perc_params}')
perceptron_cv_df.round(4)

Perceptron CV tarama süresi : 12.25 sn
Seçilen en iyi parametreler : {'eta0': 0.0001, 'class_weight': 'balanced'}


,eta0,class_weight,CV_Macro_F1,CV_Macro_F1_Std,CV_Weighted_F1,CV_ROC_AUC,CV_PR_AUC,Seçim_Skoru
0,0.0001,balanced,0.5321,0.0246,0.7990,0.6226,0.1831,0.4459
1,0.0010,balanced,0.5321,0.0246,0.7990,0.6226,0.1831,0.4459
2,0.1000,None (Ağırlıksız),0.5376,0.0258,0.8292,0.6176,0.1779,0.4444
3,0.5000,None (Ağırlıksız),0.5376,0.0258,0.8292,0.6176,0.1778,0.4443
4,1.0000,None (Ağırlıksız),0.5376,0.0258,0.8292,0.6176,0.1746,0.4433
5,0.0100,balanced,0.5066,0.0718,0.7519,0.6190,0.1829,0.4361
6,0.0500,balanced,0.5066,0.0718,0.7519,0.6190,0.1829,0.4361
7,0.1000,balanced,0.5066,0.0718,0.7519,0.6190,0.1829,0.4361
8,0.0500,None (Ağırlıksız),0.5288,0.0314,0.8064,0.6043,0.1697,0.4343
9,0.5000,balanced,0.5066,0.0718,0.7519,0.6184,0.1741,0.4331


In [4]:
y_pred_perc = best_perceptron.predict(X_test.to_numpy(dtype=float))
y_prob_perc = models.perceptron_probabilities(best_perceptron, X_test)
perc_test_metrics = models.evaluate_predictions(
    y_test, y_pred_perc, y_prob_perc
)

print('Perceptron (Baseline) X_test Performansı:')
pd.DataFrame([perc_test_metrics]).round(4)

Perceptron (Baseline) X_test Performansı:


,Macro_F1,Weighted_F1,ROC_AUC,PR_AUC
0,0.5338,0.7748,0.6114,0.1596


## 3. Model 2 — Ağaç Tabanlı GBDT (`LightGBM` / `XGBoost`) ve `Optuna` Optimizasyonu

İlk adımda varsayılan dengeli `LightGBM` ve `XGBoost` modelleri yalnızca `X_train` üzerinde **5-Katlı Stratified CV** ile karşılaştırılır; ardından `LightGBM` modeli `Optuna` (`TPESampler`) ile optimize edilir.

In [5]:
t0 = time.perf_counter()
gbdt_candidates_df = models.compare_gbdt_candidates(
    X_train, y_train, n_splits=5
)
cand_sec = time.perf_counter() - t0

print(f'Aday GBDT CV karşılaştırma süresi: {cand_sec:.2f} sn')
gbdt_candidates_df.round(4)

Aday GBDT CV karşılaştırma süresi: 14.65 sn


,Model_Adayı,CV_Macro_F1,CV_Weighted_F1,CV_ROC_AUC,CV_PR_AUC
0,LightGBM (Varsayılan - Dengeli),0.5364,0.7271,0.7063,0.2507
1,XGBoost (Varsayılan - Dengeli),0.5312,0.7199,0.7052,0.2477


### `Optuna` + `StratifiedKFold` ile Sızıntısız Hiperparametre Optimizasyonu

`X_test` veri setine hiçbir şekilde dokunulmadan, `X_train` üzerinde 5-katlı `StratifiedKFold` kullanılarak `LightGBM` modelinin ağaç derinliği, yaprak sayısı, öğrenme oranı, alt örnekleme, L1/L2 regülarizasyon ve pozitif sınıf ağırlığı (`scale_pos_weight`) hiperparametreleri optimize edilir.

In [6]:
t0 = time.perf_counter()
best_gbdt, best_gbdt_params, optuna_trials_df, optuna_study = (
    models.optimize_gbdt_with_optuna(
        X_train, y_train, n_trials=20, n_splits=5
    )
)
optuna_sec = time.perf_counter() - t0

print(f'Optuna optimizasyon süresi : {optuna_sec:.2f} sn')
print('En iyi Optuna hiperparametreleri:')
for k, v in best_gbdt_params.items():
    if k not in {'random_state', 'n_jobs', 'verbose', 'subsample_freq'}:
        print(f'  - {k}: {v}')

optuna_trials_df.head(10).round(4)

Optuna optimizasyon süresi : 297.08 sn
En iyi Optuna hiperparametreleri:
  - n_estimators: 400
  - learning_rate: 0.023100342935004098
  - num_leaves: 55
  - max_depth: 8
  - min_child_samples: 98
  - subsample: 0.8990607491281115
  - colsample_bytree: 0.5786996046075434
  - reg_alpha: 0.08878325477238118
  - reg_lambda: 4.906523775243817
  - scale_pos_weight: 4.982041289577474


,Trial,Hedef_Skor,CV_Macro_F1,CV_Weighted_F1,CV_ROC_AUC,CV_PR_AUC,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda,scale_pos_weight
0,17,0.5396,0.5983,0.8280,0.7091,0.2526,400,0.0231,55,8,98,0.8991,0.5787,0.0888,4.9065,4.9820
1,19,0.5393,0.5982,0.8351,0.7084,0.2524,400,0.0230,61,10,100,0.9226,0.6109,0.0255,4.4205,4.6635
2,13,0.5388,0.5969,0.8349,0.7087,0.2526,400,0.0271,39,8,105,0.9186,0.5051,0.0316,0.3190,4.5902
3,2,0.5386,0.5965,0.8338,0.7088,0.2525,300,0.0257,38,6,107,0.9141,0.5998,0.1140,0.2342,4.6394
4,14,0.5381,0.5959,0.8285,0.7087,0.2519,300,0.0224,28,7,109,0.9725,0.5521,0.0160,0.1622,4.8831
5,15,0.5379,0.5953,0.8230,0.7085,0.2523,400,0.0254,39,9,108,0.7757,0.6057,0.0159,0.0897,5.1737
6,18,0.5378,0.5962,0.8232,0.7072,0.2515,400,0.0360,43,8,85,0.9150,0.5129,0.0096,4.6365,5.2234
7,12,0.5377,0.5951,0.8200,0.7086,0.2521,400,0.0235,32,5,116,0.8836,0.8301,0.0669,0.0377,5.2054
8,7,0.5376,0.5951,0.8250,0.7089,0.2512,350,0.0286,15,9,150,0.8916,0.8856,0.0020,0.0272,4.8476
9,16,0.5372,0.5965,0.8376,0.7053,0.2507,350,0.0391,53,8,67,0.8536,0.5193,0.0413,0.2289,4.5229


In [7]:
pos_weight_default = float((y_train == 0).sum() / (y_train == 1).sum())
baseline_lgbm = models.build_baseline_lightgbm(
    scale_pos_weight=pos_weight_default
)
baseline_lgbm.fit(X_train.to_numpy(dtype=float), y_train.to_numpy(dtype=int))
y_pred_lgbm_base, y_prob_lgbm_base = models.predict_gbdt(
    baseline_lgbm, X_test
)

y_pred_gbdt, y_prob_gbdt = models.predict_gbdt(best_gbdt, X_test)
gbdt_test_metrics = models.evaluate_predictions(
    y_test, y_pred_gbdt, y_prob_gbdt
)

print('Optuna ile Optimize Edilmiş LightGBM X_test Performansı:')
pd.DataFrame([gbdt_test_metrics]).round(4)

Optuna ile Optimize Edilmiş LightGBM X_test Performansı:


,Macro_F1,Weighted_F1,ROC_AUC,PR_AUC
0,0.5846,0.8219,0.6744,0.2264


## 4. GBDT Özellik Önemlerinin (`feature_importances_`) Çıkarılması

Faz 5'teki **4. Zorunlu Grafik** (*TabNet attention mask visualization alongside GBDT feature importances*) için hem `154` kodlanmış özellik düzeyinde hem de `42` ana klinik özellik düzeyinde GBDT özellik önemleri çıkarılır.

In [8]:
raw_feature_cols = [
    c for c in df_missing.columns if c != config.TARGET
]
feat_imp_154 = models.extract_feature_importances(
    best_gbdt, list(X_train.columns)
)
feat_imp_42 = models.aggregate_feature_importances(
    feat_imp_154, raw_columns=raw_feature_cols
)

print('En Önemli 15 Kodlanmış Özellik (154 Sütun Uzayı):')
display(feat_imp_154.head(15).round(4))

print('\nEn Önemli 15 Ana Klinik Özellik (42 Orijinal Sütun Uzayı):')
display(feat_imp_42.head(15).round(4))

En Önemli 15 Kodlanmış Özellik (154 Sütun Uzayı):


,Özellik,Önem_Skoru,Önem_Oranı,Önem_%
0,diag_2,1617.0,0.0983,9.8262
1,diag_3,1574.0,0.0956,9.5649
2,diag_1,1546.0,0.0939,9.3947
3,num_lab_procedures,1269.0,0.0771,7.7115
4,num_medications,1122.0,0.0682,6.8182
5,time_in_hospital,915.0,0.0556,5.5603
6,number_diagnoses,784.0,0.0476,4.7642
7,medical_specialty,728.0,0.0442,4.4239
8,number_inpatient,691.0,0.0420,4.1991
9,num_procedures,682.0,0.0414,4.1444



En Önemli 15 Ana Klinik Özellik (42 Orijinal Sütun Uzayı):


,Özellik,Önem_Skoru,Önem_Oranı,Önem_%
0,diag_2,1617.0,0.0983,9.8262
1,diag_3,1574.0,0.0956,9.5649
2,diag_1,1546.0,0.0939,9.3947
3,num_lab_procedures,1269.0,0.0771,7.7115
4,discharge_disposition_id,1188.0,0.0722,7.2193
5,num_medications,1122.0,0.0682,6.8182
6,time_in_hospital,915.0,0.0556,5.5603
7,number_diagnoses,784.0,0.0476,4.7642
8,medical_specialty,728.0,0.0442,4.4239
9,number_inpatient,691.0,0.0420,4.1991


## 5. Faz 4 Model Karşılaştırması ve Faz 5 İçin Çıktıların Kaydedilmesi

Dokunulmamış gerçek test seti (`X_test`, `20.354` hasta) üzerinde referans Perceptron, varsayılan GBDT ve Optuna ile optimize edilmiş GBDT modelleri yalnızca **`Macro_F1`, `Weighted_F1`, `ROC_AUC` ve `PR_AUC`** metrikleriyle karşılaştırılır ve Faz 5 için `data/processed/faz4_outputs.pkl` dosyasına kaydedilir.

In [9]:
test_models = {
    'Perceptron (Baseline - Tuned eta0)': {
        'y_pred': y_pred_perc,
        'y_prob': y_prob_perc,
    },
    'LightGBM (Baseline - Varsayılan)': {
        'y_pred': y_pred_lgbm_base,
        'y_prob': y_prob_lgbm_base,
    },
    'LightGBM (Optuna + 5-Fold Stratified CV)': {
        'y_pred': y_pred_gbdt,
        'y_prob': y_prob_gbdt,
    },
}

comparison_df = models.evaluate_models_on_test(test_models, y_test)
assert 'Accuracy' not in comparison_df.columns

faz4_outputs = {
    'y_test': y_test,
    'feature_names_154': list(X_train.columns),
    'feature_names_42': raw_feature_cols,
    'perceptron': {
        'model': best_perceptron,
        'best_params': best_perc_params,
        'cv_tuning_df': perceptron_cv_df,
        'y_pred': y_pred_perc,
        'y_prob': y_prob_perc,
        'metrics': perc_test_metrics,
        'confusion_matrix_norm': models.normalized_confusion_matrix(
            y_test, y_pred_perc
        ),
    },
    'gbdt': {
        'model': best_gbdt,
        'best_params': best_gbdt_params,
        'optuna_trials_df': optuna_trials_df,
        'candidates_cv_df': gbdt_candidates_df,
        'y_pred': y_pred_gbdt,
        'y_prob': y_prob_gbdt,
        'metrics': gbdt_test_metrics,
        'confusion_matrix_norm': models.normalized_confusion_matrix(
            y_test, y_pred_gbdt
        ),
        'feature_importances': feat_imp_154,
        'feature_importances_grouped': feat_imp_42,
    },
    'comparison_df': comparison_df,
}

saved_path = models.save_faz4_outputs(faz4_outputs)
faz5_payload = models.get_faz5_plot_payload()
print(f'Faz 4 çıktıları kaydedildi: {saved_path.name}')
print(f"Faz 5 grafik modelleri    : {list(faz5_payload['models'].keys())}")
comparison_df.round(4)

Faz 4 çıktıları kaydedildi: faz4_outputs.pkl
Faz 5 grafik modelleri    : ['Perceptron (Baseline)', 'LightGBM (Optuna GBDT)']


,Model,Macro_F1,Weighted_F1,ROC_AUC,PR_AUC
0,Perceptron (Baseline - Tuned eta0),0.5338,0.7748,0.6114,0.1596
1,LightGBM (Baseline - Varsayılan),0.5237,0.7159,0.6739,0.2245
2,LightGBM (Optuna + 5-Fold Stratified CV),0.5846,0.8219,0.6744,0.2264
